# ZTE — the attention read-out

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/tbme/zte_attention.ipynb)

**Upload this notebook on its own.** It clones the repo, reads your Drive, and produces the two physiological
figures a reviewer asks for: *when* in the word the encoder's own attention lands, and *where* on the scalp.
Nothing is retrained. The trained checkpoint is run over its held-out subject's readings in evaluation mode with
PyTorch forward hooks on its attention modules, and the captured weights are averaged, plotted and mapped.

| § | What | Reads | Roughly |
| --- | --- | --- | --- |
| **4a** | Is the checkpoint's geometry real? | the suite's ZAB sentence arm, straight from its `best.pt` buffers | seconds |
| **4b** | Retrain that fold with the montage present — only if 4a finds the placeholder | `experiments/alignment/sentence/combined.yaml` | ~40 min on an A100 |
| **4** | The hooked pass over one held-out subject | the checkpoint 4a chose | ~5 min on a GPU, longer on CPU |
| **5** | The figures and the numbers | §4's `attention.json` | seconds |
| **6** | Every fold — optional | the eleven other sentence-level holdouts on Drive | ~5 min each |
| **7** | The paper's figures, from the real artifacts | §4's `attention.json` and the suite's `PARALLAX.json` | ~1 min |

**The scalp map is drawn on the head the model was trained on, or not at all.** The lens uses coordinates only
when they provably rebuild the checkpoint's own spherical-harmonic basis — the CSV the run named, the persistent
store's copy, or the ZuCo-105 montage shipped inside the package — so a fresh VM without `res/montage_gsn105.csv`
no longer reports an exact checkpoint as approximate. §4a reads the answer off the checkpoint before anything runs.

**This is inspection, not a result.** Attention weights describe what the model computed, not why its output
moved; the counterfactual instrument is the occlusion profile in `zte_tbme.ipynb` §12, and this notebook is read
beside it. Every artifact here carries the lens disclaimer and gates nothing.

`RESUME_DATE` in §3 is preset to `2026-08-29`, the evidence-suite session that trained the twelve sentence-level
folds. Point it elsewhere only to read a different set of weights.

## 1 · Provision the runtime

Installs `uv`, clones or refreshes the repo, and builds the pinned Python 3.14 venv that every `!uv run` below uses.
The kernel you are typing in is Colab's own older interpreter and never imports `zte`.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

`colab()` is the only route into the package: it runs one `zte-colab` subcommand in the venv and returns the JSON it
printed. Nothing below computes with ZTE in this kernel — it renders payloads the venv produced.

In [ ]:
import json
import os
import pathlib
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

plan, res = ENV['plan'], ENV['resources']
gpu = f'{res["gpu"]["name"]} ({res["gpu"]["total_gb"]} GB)' if res.get('gpu') else 'none - the hooked pass runs on CPU'
print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} - zte {ENV["venv"]["zte"]}   <- every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   <- this cell; renders payloads, never imports zte')
print(f'device : {plan["device"]}  -  GPU {gpu}')

## 3 · Drive is the workspace

The checkpoints this notebook reads were trained by `zte_tbme.ipynb` and live under the dated session folder on
Drive. `RESUME_DATE` names that folder; leaving it `None` would open a folder named by *today's* date, which holds
no checkpoints, and `resolve_ckpt` would fail loudly rather than guess.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# The session whose checkpoints are read. 2026-08-29 is the evidence-suite session that trained the twelve
# sentence-level folds; a different date reads different weights, so it is fixed here rather than defaulted to
# today, which would open an empty folder in which nothing can be found.
RESUME_DATE: str | None = '2026-08-29'
WRITE_MODE: str = 'auto'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
LOCAL_RUNS: str = SESSION['local_runs']
OUT_ROOT: str = SESSION['out_root']
DRIVE_BACKUP: str = SESSION['drive_backup']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

# The same holdout and seed as the evidence suite, because the run name they form is what is read.
SEED: int = 42
HOLDOUT: str = 'ZAB'
# True trains the montage-present fold in section 4b whatever section 4a finds, and reads that fold from then on.
# The verdict 4a prints is still the checkpoint's own; this only decides whether to trust it or retrain regardless.
FORCE_RETRAIN: bool = False

# Only section 4b trains, and only when 4a finds the placeholder cap. These are the evidence suite's own flags, so
# the retrained fold differs from the suite's ZAB arm in nothing but the montage being present when it is built.
TRAIN_FLAGS: str = (
    f'--seed {SEED} --data-cache "{PREPARED_LOCAL}" --data-cache-remote "{PREPARED_DRIVE}" '
    f'--drive-backup "{DRIVE_BACKUP}" --spatial exact'
)

# Lands beside the evidence suite's other audits, under the Drive session, so the paper reads one tree. Without a
# Drive mount the same layout goes to the local tree and only the Drive copy is lost.
SUITE: str = f'{DRIVE_ANALYSIS}/evidence_suite' if SESSION['drive_mounted'] else 'res/evidence_suite'
ATTENTION_OUT: str = f'{SUITE}/attention'

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "NEW - it holds no checkpoints"})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}')
print(f'writes -> : {ATTENTION_OUT}')
if not SESSION['resumed']:
    print('\nThis session is new. Set RESUME_DATE to the session that trained the sentence-level folds.')
if not SESSION['data_dir_present']:
    print('\nZuCo is not at that path. The hooked pass reads the corpus and will fail until it is.')

### 3a · Helpers

`resolve_ckpt` finds a run's `best.pt` on Drive first, so a fresh VM can read a session it did not train; it never
falls back to `last.pt`, which is a different model. `audit()` and `show()` read the JSON and Markdown the CLI
wrote — nothing is recomputed in this kernel.

In [ ]:
def resolve_ckpt(run_name: str, which: str = 'best') -> str:
    """Finds a run's checkpoint, Drive first, so a fresh VM can read a session it did not train.

    A missing `best.pt` never falls back to `last.pt`: they are different models, and swapping them silently
    misattributes the number.
    """
    for run in colab('runs', '--drive', ZTE_DRIVE, '--experiments', LOCAL_RUNS, '--run', run_name)['runs']:
        if path := run['checkpoints'][which]:
            print(f'{which}.pt for {run_name}: {"Drive" if run["source"] == "drive" else "local disk"}\n  {path}')
            return path

    raise FileNotFoundError(f'no {which}.pt for {run_name!r} on Drive or locally; train it first.')


def audit(kind: str, directory: str, markdown: bool = True) -> dict[str, Any]:
    """Reads one audit's JSON (and its Markdown) out of a directory, recomputing nothing."""
    flags = [] if markdown else ['--no-markdown']
    payload = colab('audit', '--from', directory, '--kind', kind, *flags)['audits'][kind]
    if not payload['found']:
        print(f'no {kind} artifact at {payload["json_path"]} - run the cell above it first.')

    return payload


def show(payload: dict[str, Any]) -> None:
    """Renders an audit's own Markdown inline, so the notebook shows the report the CLI wrote."""
    from IPython.display import Markdown, display

    if payload.get('markdown'):
        display(Markdown(payload['markdown']))
    else:
        print('no rendered Markdown beside that artifact.')

In [ ]:
# Rendering only: these read the JSON and the PNGs that `zte-lens attention` already produced.
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Image, display

## 4 · The hooked pass

`zte-lens attention` loads the checkpoint, builds its held-out subject's readings, and registers PyTorch forward
hooks (`register_forward_pre_hook` / `register_forward_hook`) on two modules of the frozen encoder, in `eval()`
mode under `no_grad`:

| module | attends over | weights per word | what is kept |
| --- | --- | --- | --- |
| `SpatialChannelMixer.attn` — the electrode mixer | the 105 electrodes, each a key described by its whole 350-sample trace | `(heads, 105, 105)` | attention **received** per electrode: mean over heads and queries |
| `RawConformer.transformer` — the intra-word transformer | the 350 time steps (700 ms at 500 Hz) of the word window | `(layers, heads, 350, 350)` | attention received per time step, per layer |

**Why *received*, and why the last layer.** Every live arm pools the intra-word transformer's output by a plain mean
over time, so the word vector is the average over query positions $q$ of the attended values $v_k$:

$$
h \;=\; \frac{1}{T}\sum_{q}\sum_{k} A_{qk}\, v_k \;=\; \sum_{k}\Big(\frac{1}{T}\sum_{q} A_{qk}\Big)\, v_k \;=\; \sum_k a_k\, v_k
$$

so $a_k$, the column mean of the last layer's attention matrix $A$, is exactly the weight time step $k$'s value
carries into the word. It sums to one over the window, and a uniform profile sits at $1/T = 1/350$. Every curve is
averaged over a reading's words and then bootstrapped over readings.

**Which readings.** Each of the subject's readings is scored against the *other* subjects' readings exactly as
`held_out_retrieval` scores it — cosine, the query's own subject excluded from the gallery. A reading whose sentence
ranks first is `correct`, the rest are `incorrect`, and `all` is reported too. At Top-1 on ~700 stimuli the correct
set is small (the ZAB sentence arm retrieved 19) and the bootstrap interval says how small; `--correct-top-k 10`
widens it. This selection is never a retrieval result — the scoreboard and the length audit own that number, and the
selection here is unstratified with no post-processing.

**Two things the scalp map cannot say.** The mixer's weights carry **no latency axis**: each electrode is one key
described by its entire 700 ms trace, so the topography is attention received across the whole word, and the N400
band applies to the temporal curve alone. And the map is drawn only when the checkpoint's own `approximate_geometry`
buffer reads `False` *and* a montage on this VM provably rebuilds its harmonic basis — §4a checks both before
anything runs — because on the coordinate-free cap a topoplot shows array indices, not regions, and on the wrong
montage it shows the wrong head.

**The caveat every artifact carries.** Attention weights describe what the model computed, not why its output moved.
ZuCo's eye-tracking segmentation makes a word window overlap its neighbours, so a peak in 300–500 ms is *consistent
with* an N400 and is not evidence of one. `peak_in_n400_window` gates nothing.

### 4a · Is the checkpoint's geometry real?

The answer lives in the checkpoint, not in the YAML and not in this VM's files: `harmonics`, `degrees` and
`approximate` are persistent buffers beside the electrode code, so they describe the head the numbers were computed
on wherever the weights are loaded. `zte-colab geometry` reads them directly and then asks whether any montage on
this machine — the CSV the run named, the persistent store's copy, or the ZuCo-105 montage shipped inside the
package — rebuilds that exact basis. An earlier release of the lens read a flag frozen when the model was *built*,
so on a VM without `res/montage_gsn105.csv` an exact checkpoint was reported as approximate and its scalp map
declined; this cell is what decides, from the buffers, whether §4b is needed at all.

In [ ]:
SUITE_RUN = f'align_sentence_combined_lo{HOLDOUT}_s{SEED}'
# The fold retrained with the montage present, if 4a finds the placeholder. Deliberately not `_combined_`: the level
# audit and the LOSO summary sweep that pattern, and a thirteenth sentence-level fold must not enter their tables.
GSN_RUN = f'align_sentence_gsn105_lo{HOLDOUT}_s{SEED}'

GEOMETRY = colab('geometry', '--ckpt', resolve_ckpt(SUITE_RUN))
basis = 'placeholder cap' if GEOMETRY['approximate_geometry'] else 'exact montage'
print(f'run      : {GEOMETRY["run_name"]}   holdout {GEOMETRY["train_holdout"]}')
print(f'basis    : {basis}   ({GEOMETRY["n_channels"]} electrodes, degree {GEOMETRY["l_max"]})')
print(f'montage  : {GEOMETRY["montage_source"] or "none verified"}   {GEOMETRY["montage_path"] or ""}')
print(f'readable : {GEOMETRY["topomap_readable"]}   {GEOMETRY["reason"] or ""}')

ATTENTION_RUN = GSN_RUN if FORCE_RETRAIN or not GEOMETRY['topomap_readable'] else SUITE_RUN
if FORCE_RETRAIN:
    print(f'\nFORCE_RETRAIN is on: section 4b trains {GSN_RUN} whatever the verdict above, and section 4 reads it.')
elif GEOMETRY['approximate_geometry']:
    print(f'\nThe suite checkpoint was trained on the placeholder cap. Run section 4b: it trains {GSN_RUN}.')
elif not GEOMETRY['topomap_readable']:
    raise SystemExit(f'The basis is exact but no montage on this VM reproduces it: {GEOMETRY["reason"]}')
else:
    print(f'\nExact basis, verified montage. Section 4b is not needed; section 4 reads {SUITE_RUN}.')

### 4b · Retrain one fold with the montage present

Only when §4a found the placeholder, or `FORCE_RETRAIN` in §3 is `True`. A trained basis cannot be repaired by a flag, and `--resume` under the suite's
run name would restore the placeholder buffers and finish instantly, so the fold is trained under a new name from the
same config, seed, holdout and flags. The one difference is that the montage exists when the encoder is built:
`--spatial exact` provisions it from the persistent store, from `mne`, or from the copy shipped inside the package,
and now refuses to train on the cap rather than degrade to it. The suite's sentence folds took about 2.2 ks each on
an A100. The cell is idempotent: a finished run is skipped, an interrupted one resumes.

In [ ]:
if ATTENTION_RUN == GSN_RUN:
    print(f'===== {GSN_RUN} =====')
    !uv run zte-run --config experiments/alignment/sentence/combined.yaml --root "{DATA_DIR}" --name "{GSN_RUN}" \
        --loso-holdout {HOLDOUT} --out-root "{OUT_ROOT}" {TRAIN_FLAGS} --resume
    GEOMETRY = colab('geometry', '--ckpt', resolve_ckpt(GSN_RUN))
    if not GEOMETRY['topomap_readable']:
        raise SystemExit(f'{GSN_RUN} still has no readable geometry: {GEOMETRY["reason"]}')
    print(f'{GSN_RUN}: exact basis, montage verified from {GEOMETRY["montage_source"]}.')
else:
    print('Not needed: section 4a found an exact basis and a verified montage.')

### 4c · Run it

In [ ]:
ATTENTION_CKPT = resolve_ckpt(ATTENTION_RUN)
ATTENTION_DIR = f'{ATTENTION_OUT}/{ATTENTION_RUN}_{HOLDOUT}_attention'

# `--batch-size 4`: the per-head weights are (words x heads x 350 x 350) per batch, so it stays small on purpose.
# A done stamp beside attention.json skips an identical re-run, so this cell is safe to repeat; a profile written
# before the lens verified its montage carries an older schema and is rebuilt rather than served.
!uv run zte-lens attention --ckpt "{ATTENTION_CKPT}" --root "{DATA_DIR}" --correct-top-k 1 --batch-size 4 \
    --out "{ATTENTION_OUT}"

## 5 · Read it

The CLI wrote three things into `ATTENTION_DIR`: `attention.json` (every curve, interval and electrode), `attention.md`
(the summary rendered below), and the two figures, each as a PNG and a vector PDF —
`attention_temporal.*` from matplotlib and `attention_topomap.*` from `mne.viz.plot_topomap`.

In [ ]:
show(audit('attention', ATTENTION_DIR))

In [ ]:
ATT = audit('attention', ATTENTION_DIR, markdown=False)['payload']
if not ATT:
    raise SystemExit('No attention profile yet -- run the cell in section 4 before this one.')

COLOURS = {'correct': '#c0392b', 'incorrect': '#7f8c8d', 'all': '#2c3e50'}
temporal = ATT.get('temporal') or {}
if temporal:
    layer = temporal['n_layers'] - 1
    times = temporal['times_ms']
    fig = go.Figure()
    for name in ('correct', 'incorrect', 'all'):
        block = temporal['groups'].get(name)
        if not block:
            continue
        curve = block['layers'][layer]
        fig.add_scatter(
            x=times + times[::-1],
            y=curve['ci_high'] + curve['ci_low'][::-1],
            fill='toself',
            fillcolor=COLOURS[name],
            opacity=0.15,
            line={'width': 0},
            showlegend=False,
            hoverinfo='skip',
        )
        fig.add_scatter(
            x=times,
            y=curve['mean'],
            mode='lines',
            name=f'{name} (n={block["n_readings"]} readings, {block["n_words"]} words)',
            line={'color': COLOURS[name]},
        )
    lo, hi = temporal['n400_window_ms']
    fig.add_vrect(x0=lo, x1=hi, fillcolor='orange', opacity=0.12, line_width=0)
    fig.add_hline(y=temporal['uniform'], line_dash='dot', line_color='grey', annotation_text='uniform attention')
    fig.update_layout(
        title=f'Attention received per time step, layer {layer} of {temporal["n_layers"]} - subject {ATT["subject"]}',
        xaxis_title='ms from word onset',
        yaxis_title='attention received (mean over heads and queries)',
        height=440,
    )
    fig.show()

    sel = ATT['selection']
    print(f'readings profiled : {sel["n_profiled"]}   correct at {sel["criterion"]}: {sel["n_correct"]}', end='')
    print(f'   chance Top-1 {sel["chance_top1"]:.4f}   post-processing {sel["postprocess_fit"]}')
    for name in ('correct', 'incorrect', 'all'):
        block = temporal['groups'].get(name)
        if not block:
            continue
        lo_ci, hi_ci = block['n400_mass_ci']
        print(f'{name:<10} N400 mass {block["n400_mass"]:.4f} [{lo_ci:.4f}, {hi_ci:.4f}]', end='')
        print(f'   uniform {block["n400_mass_uniform"]:.4f}   peak {block["peak_ms"]:.0f} ms', end='')
        print(f'   in band: {block["peak_in_n400_window"]}')
    contrast = temporal['contrast']
    if contrast['n400_mass_difference'] is not None:
        d_lo, d_hi = contrast['n400_mass_difference_ci']
        print(f'\ncorrect - incorrect N400 mass: {contrast["n400_mass_difference"]:+.4f} [{d_lo:+.4f}, {d_hi:+.4f}]')
        print('An interval containing zero means the retrieved readings were not attended differently in the band.')
else:
    print(f'No temporal profile: {ATT.get("absent", {}).get("temporal")}')

In [ ]:
figures = ATT.get('figures') or {}
spatial = ATT.get('spatial') or {}

if figures.get('topomap'):
    display(Image(filename=figures['topomap']))
else:
    print(f'No scalp map: {figures.get("topomap_reason") or ATT.get("absent", {}).get("spatial")}')

if spatial:
    if spatial['approximate_geometry']:
        geometry = 'approximate cap - array indices, not regions'
    elif spatial.get('montage_verified'):
        geometry = f'exact montage, verified ({spatial["montage_source"]}: {spatial["montage_path"]})'
    else:
        geometry = f'exact montage, unmapped - {spatial.get("montage_reason")}'
    print(f'geometry : {geometry}   channels {spatial["n_channels"]}   heads {spatial["n_heads"]}')
    print('uniform  : {:.4f} per electrode'.format(1.0 / spatial['n_channels']))
    for name in ('correct', 'incorrect'):
        block = spatial['groups'].get(name)
        if not block:
            continue
        top = ', '.join(f'{c["label"]} {c["mean"]:.4f}' for c in block['top_channels'][:8])
        print(f'{name:<10} most attended: {top}')
        if block.get('region_mass'):
            regions = ', '.join(f'{r} {v:.3f}' for r, v in block['region_mass'].items())
            print(f'{"":<10} by region    : {regions}')

if figures.get('temporal'):
    display(Image(filename=figures['temporal']))

if spatial and not spatial.get('montage_verified'):
    print('=' * 100)
    print('NO SCALP FIGURE CAN BE READ FROM THIS PROFILE:', spatial.get('montage_reason'))
    print('Re-run section 4a; a placeholder basis needs section 4b, an unverified montage needs `--spatial exact`.')
    print('=' * 100)

**How to read that.** Two numbers carry the temporal figure: the **N400 mass** — the share of the last layer's
received attention inside 300–500 ms, against the $100/350 = 0.286$ a uniform profile would put there — and its
bootstrap interval over readings. The **correct − incorrect** contrast says whether the retrieved readings were
attended differently in that band; an interval excluding zero is a difference in *attention*, not a cause of the
retrieval. The scalp map is read against the uniform $1/105$ per electrode and, when the montage is exact, by
region.

What none of it licenses: a sentence that begins "the network attends to the N400". The honest sentence is "the
last-layer attention received peaks at *X* ms, with *Y* of its mass in 300–500 ms [CI]; occlusion (§12 of the
evidence suite) places the causal contribution at *Z* ms". Report both instruments, and report a flat profile as
plainly as a peaked one.

## 6 · Every fold — optional

One held-out subject is a sample of one. Every sentence-level fold of the evidence suite is on Drive under the same
session, so the same pass over all twelve gives the population version: twelve `correct` curves and the spread of
their N400 mass. Each fold costs about what §4 did, and a fold already profiled returns in the time it takes to hash
its checkpoint.

In [ ]:
FOLDS = ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']

for holdout in FOLDS:
    run = f'align_sentence_combined_lo{holdout}_s{SEED}'
    try:
        ckpt = resolve_ckpt(run)
    except FileNotFoundError as exc:
        print(f'skip {holdout}: {exc}')
        continue
    !uv run zte-lens attention --ckpt "{ckpt}" --root "{DATA_DIR}" --correct-top-k 1 --batch-size 4 \
        --out "{ATTENTION_OUT}"

In [ ]:
rows: list[dict[str, Any]] = []
axis: dict[str, Any] = {}
fig = go.Figure()
for holdout in FOLDS:
    run = f'align_sentence_combined_lo{holdout}_s{SEED}'
    payload = audit('attention', f'{ATTENTION_OUT}/{run}_{holdout}_attention', markdown=False)['payload']
    block = (((payload or {}).get('temporal') or {}).get('groups') or {}).get('correct')
    if not block:
        continue
    axis = axis or payload['temporal']
    curve = block['layers'][-1]
    scalp = payload.get('spatial') or {}
    fig.add_scatter(x=axis['times_ms'], y=curve['mean'], mode='lines', name=f'{holdout} (n={block["n_readings"]})')
    rows.append(
        {
            'holdout': holdout,
            'correct readings': block['n_readings'],
            'words': block['n_words'],
            'N400 mass': round(block['n400_mass'], 4),
            'CI': [round(v, 4) for v in block['n400_mass_ci']],
            'peak ms': block['peak_ms'],
            'in band': block['peak_in_n400_window'],
            'basis': 'placeholder' if scalp.get('approximate_geometry', True) else 'exact',
            'montage': scalp.get('montage_source') or '-',
        }
    )

if rows:
    lo, hi = axis['n400_window_ms']
    fig.add_vrect(x0=lo, x1=hi, fillcolor='orange', opacity=0.12, line_width=0)
    fig.add_hline(y=axis['uniform'], line_dash='dot', line_color='grey', annotation_text='uniform attention')
    fig.update_layout(
        title='Attention received per time step, correctly retrieved readings, one curve per held-out subject',
        xaxis_title='ms from word onset',
        yaxis_title='attention received (last layer)',
        height=460,
    )
    fig.show()

    frame = pd.DataFrame(rows)
    display(frame)
    masses = frame['N400 mass']
    uniform = axis['groups']['correct']['n400_mass_uniform']
    print(f'N400 mass over {len(frame)} folds: {masses.mean():.4f} +/- {masses.std(ddof=1):.4f} (sample sd)', end='')
    print(f'   uniform {uniform:.4f}')
else:
    print('No fold has an attention profile yet - run the cell above.')

## 7 · The paper's figures, from the real artifacts

`zte-schematics` draws the method schematics — the encoder, the objective, the electrode geometry, the decoder and
the data, twenty-nine in all with variants to choose between (`docs/RUNNING.md` lists them) — and, from what this
session produced, the three figures that must come from real data: the attention scalp map and temporal curve from
§4's `attention.json`, and the cross-task transfer heatmap from the evidence suite's `PARALLAX.json`. Every figure
is written as PNG at 300 dpi and SVG at IEEE column widths, with a `contact_sheet.png` to pick from.

The scalp map is refused unless the montage recorded in `attention.json` was verified against the checkpoint's own
basis, so an unreadable profile cannot become a figure by accident. The transfer heatmap needs the transfer study to
have run in this session (`zte_tbme.ipynb` §7); without it the cell says so and draws everything else.

In [ ]:
FIGURES_OUT = f'{SUITE}/schematics'
ATTENTION_JSON = f'{ATTENTION_DIR}/attention.json'
# The transfer matrix belongs to the evidence suite; it exists only if zte_tbme.ipynb section 7 ran in this session.
PARALLAX_JSON = f'{SUITE}/transfer/PARALLAX.json'
PARALLAX_FLAG = f'--parallax "{PARALLAX_JSON}"' if pathlib.Path(PARALLAX_JSON).is_file() else ''
if not PARALLAX_FLAG:
    print(f'no {PARALLAX_JSON}: the transfer heatmap is skipped; run zte_tbme.ipynb section 7 in this session first.')

!uv run zte-schematics --out "{FIGURES_OUT}" --attention "{ATTENTION_JSON}" {PARALLAX_FLAG}

for name in ('attention_topomap', 'attention_temporal', 'transfer_heatmap', 'contact_sheet'):
    png = pathlib.Path(FIGURES_OUT) / f'{name}.png'
    if png.is_file():
        display(Image(filename=str(png)))
print(f'figures -> {FIGURES_OUT}')

## 8 · Where it landed, and the download

Everything above was written straight to Drive when it was mounted, so a reclaimed VM costs nothing. Each
`<run>_<subject>_attention/` directory holds `attention.json`, `attention.md`, and the lens's two figures as a PNG and
a vector PDF (`attention_temporal.*`, `attention_topomap.*`); `schematics/` holds the paper's figures as PNG and SVG,
with `attention.json` as the provenance every caption is written from.

The cell below zips the `attention/` tree — every fold that has been profiled — together with `schematics/` into the
session on Drive and hands the archive to the browser as a download.

In [ ]:
import zipfile

# One archive of every attention artifact and every figure: JSON, Markdown, PNG, PDF and SVG, for every fold
# profiled so far. It sits on Drive beside the trees it packs, so a lost download is one click away, not a re-run.
ARCHIVE = f'{SUITE}/attention_{RUN_DATE}.zip'
packed: list[str] = []
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for tree in ('attention', 'schematics'):
        root = pathlib.Path(SUITE) / tree
        if not root.is_dir():
            continue
        for file in sorted(p for p in root.rglob('*') if p.is_file()):
            archive.write(file, arcname=str(file.relative_to(SUITE)))
            packed.append(str(file.relative_to(SUITE)))

print(f'archive   : {ARCHIVE}   ({os.path.getsize(ARCHIVE) / 1e6:.1f} MB, {len(packed)} files)')
for name in packed:
    print(f'  {name}')

print(f'\nattention : {ATTENTION_OUT}')
print(f'figures   : {SUITE}/schematics')
print(f'session   : {RUN_DATE}')
print(f'To read these again on a new VM: keep RESUME_DATE = {RUN_DATE!r} in section 3 and run from section 1.')

try:
    from google.colab import files  # type: ignore[import-untyped]

    files.download(ARCHIVE)
except Exception as exc:  # not on Colab, or the browser refused: the archive is still on Drive at the path above
    print(f'\nbrowser download unavailable ({type(exc).__name__}); fetch the archive from Drive instead.')